In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-04 18:52:43,688]::2853905153::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:
best_model_path = './save_swintransformer_folder/model_best_multiseed.pth'
os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
print(1)


1


In [6]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [7]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [8]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [9]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [10]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [11]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0], fullshot=train_set_data[1], tagdata=train_set_data[2], transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0], fullshot=validation_set_data[1], tagdata=validation_set_data[2], transform=transform)
print(1)


1


In [12]:
criterion = torch.nn.L1Loss().to(device)
print(1)


1


In [13]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [14]:
SEEDS = [42, 123, 777, 1004, 2024]

patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_results = []
global_best_mae = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(2):

    for seed_idx, seed in enumerate(SEEDS, start=1):

        logger.info(
            '================ Seed {}/{}: {} ================'.format(
                seed_idx, len(SEEDS), seed
            )
        )

        set_seed(seed)

        # ----------------------------------------------------
        # Rebuild DataLoaders for every seed
        # ----------------------------------------------------
        train_generator = torch.Generator()
        train_generator.manual_seed(seed)

        val_generator = torch.Generator()
        val_generator.manual_seed(seed + 10000)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=train_generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=val_generator
        )

        # ----------------------------------------------------
        # Fresh model / optimizer for every seedc
        # ----------------------------------------------------
        model = SwinTransformer3D()
        model.to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr
        )

        seed_best_mae = float('inf')
        seed_best_epoch = 0
        no_improve_count = 0

        for epoch in range(epochs):

            # ==================================================
            # Train
            # ==================================================
            model.train()

            train_loss_sum = 0.0

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc='Seed {} Train Epoch {}'.format(seed, epoch + 1)
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                audiodata = audiodata.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(
                    fullshot,
                    audiodata
                )

                loss = criterion(
                    hypothesis,
                    big_five_data
                )

                loss.backward()
                optimizer.step()

                train_loss_sum += loss.item()

            train_avg_loss = train_loss_sum / len(train_dataloader)

            # ==================================================
            # Validation
            # ==================================================
            model.eval()

            val_loss_sum = 0.0
            val_o_sum = 0.0
            val_c_sum = 0.0
            val_e_sum = 0.0
            val_a_sum = 0.0
            val_n_sum = 0.0

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc='Seed {} Val Epoch {}'.format(seed, epoch + 1)
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(
                        fullshot,
                        audiodata
                    )

                    val_loss = criterion(
                        hypothesis,
                        big_five_data
                    )

                    (
                        hypothesiso,
                        hypothesisc,
                        hypothesise,
                        hypothesisa,
                        hypothesisn
                    ) = extract_ocean(
                        hypothesis,
                        len(fullshot)
                    )

                    (
                        big_five_datao,
                        big_five_datac,
                        big_five_datae,
                        big_five_dataa,
                        big_five_datan
                    ) = extract_ocean(
                        big_five_data,
                        len(fullshot)
                    )

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_loss_sum += val_loss.item()
                    val_o_sum += val_o_loss.item()
                    val_c_sum += val_c_loss.item()
                    val_e_sum += val_e_loss.item()
                    val_a_sum += val_a_loss.item()
                    val_n_sum += val_n_loss.item()

            val_avg_loss = val_loss_sum / len(val_dataloader)
            val_avg_o_loss = val_o_sum / len(val_dataloader)
            val_avg_c_loss = val_c_sum / len(val_dataloader)
            val_avg_e_loss = val_e_sum / len(val_dataloader)
            val_avg_a_loss = val_a_sum / len(val_dataloader)
            val_avg_n_loss = val_n_sum / len(val_dataloader)

            # ==================================================
            # Per-seed best + patience
            # ==================================================
            if val_avg_loss < seed_best_mae:

                seed_best_mae = val_avg_loss
                seed_best_epoch = epoch + 1
                no_improve_count = 0

                # Save ONLY when this is also the best model
                # across all seeds. The same file is overwritten,
                # therefore only one checkpoint remains on disk.
                if val_avg_loss < global_best_mae:

                    global_best_mae = val_avg_loss
                    global_best_seed = seed
                    global_best_epoch = epoch + 1

                    torch.save(
                        {
                            'seed': seed,
                            'epoch': epoch + 1,
                            'model': model.state_dict(),
                            'optimizer': optimizer.state_dict(),
                            'val_mae': val_avg_loss,
                            'val_1_mae': 1.0 - val_avg_loss
                        },
                        best_model_path
                    )

                    logger.info(
                        'Global best checkpoint updated: '
                        'seed={}, epoch={}, MAE={:.6f}, 1-MAE={:.6f}'.format(
                            seed,
                            epoch + 1,
                            val_avg_loss,
                            1.0 - val_avg_loss
                        )
                    )

            else:
                no_improve_count += 1

            current_1_mae = 1.0 - val_avg_loss
            seed_best_1_mae = 1.0 - seed_best_mae

            logger.debug(
                'Seed: {} , Epoch: {} , '
                '1-MAE: {:.4f} , seed best 1-MAE: {:.4f} , '
                'patience: {}/{} , '
                '1-Training Loss: {:.4f} , '
                '1-val_avg_o_loss: {:.4f} , '
                '1-val_avg_c_loss: {:.4f} , '
                '1-val_avg_e_loss: {:.4f} , '
                '1-val_avg_a_loss: {:.4f} , '
                '1-val_avg_n_loss: {:.4f}'.format(
                    seed,
                    epoch + 1,
                    current_1_mae,
                    seed_best_1_mae,
                    no_improve_count,
                    patience,
                    1.0 - train_avg_loss,
                    1.0 - val_avg_o_loss,
                    1.0 - val_avg_c_loss,
                    1.0 - val_avg_e_loss,
                    1.0 - val_avg_a_loss,
                    1.0 - val_avg_n_loss
                )
            )

            if no_improve_count >= patience:

                logger.info(
                    'Early stopping: seed={} at epoch {}. '
                    'Best epoch={}, best MAE={:.6f}, '
                    'best 1-MAE={:.6f}'.format(
                        seed,
                        epoch + 1,
                        seed_best_epoch,
                        seed_best_mae,
                        1.0 - seed_best_mae
                    )
                )

                break

            torch.cuda.empty_cache()

        # ----------------------------------------------------
        # Store only scalar results for this seed
        # ----------------------------------------------------
        seed_results.append(
            {
                'seed': seed,
                'best_epoch': seed_best_epoch,
                'best_mae': float(seed_best_mae),
                'best_1_mae': float(1.0 - seed_best_mae)
            }
        )

        logger.info(
            'Seed {} finished: best_epoch={}, '
            'best MAE={:.6f}, best 1-MAE={:.6f}'.format(
                seed,
                seed_best_epoch,
                seed_best_mae,
                1.0 - seed_best_mae
            )
        )

        # ----------------------------------------------------
        # Release GPU / CPU references before next seed
        # ----------------------------------------------------
        del model
        del optimizer
        del train_dataloader
        del val_dataloader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


# ============================================================
# Final multi-seed statistics
# ============================================================
best_1_mae_scores = np.array(
    [result['best_1_mae'] for result in seed_results],
    dtype=np.float64
)

best_mae_scores = np.array(
    [result['best_mae'] for result in seed_results],
    dtype=np.float64
)

mean_1_mae = best_1_mae_scores.mean()
std_1_mae = best_1_mae_scores.std(ddof=1)

mean_mae = best_mae_scores.mean()
std_mae = best_mae_scores.std(ddof=1)

print('\n' + '=' * 70)
print('MULTI-SEED RESULTS')
print('=' * 70)

for result in seed_results:
    print(
        'Seed {} | Best Epoch {:3d} | '
        'MAE {:.6f} | 1-MAE {:.6f}'.format(
            result['seed'],
            result['best_epoch'],
            result['best_mae'],
            result['best_1_mae']
        )
    )

print('-' * 70)
print(
    'MAE    : {:.6f} ± {:.6f}'.format(
        mean_mae,
        std_mae
    )
)
print(
    '1-MAE  : {:.6f} ± {:.6f}'.format(
        mean_1_mae,
        std_1_mae
    )
)
print(
    'Global best checkpoint: seed={}, epoch={}, '
    'MAE={:.6f}, 1-MAE={:.6f}'.format(
        global_best_seed,
        global_best_epoch,
        global_best_mae,
        1.0 - global_best_mae
    )
)
print(
    'Saved checkpoint: {}'.format(
        best_model_path
    )
)
print('=' * 70)


[2026-09-04 18:52:54,060]::3816500081::INFO::================ Seed 1/5: 42 ================


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 42 Train Epoch 1: 100%|█████████▉| 1497/1498 [06:35<00:00,  3.89it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Seed 42 Val Epoch 1: 100%|██████████| 498/498 [00:45<00:00, 10.89it/s]
[2026-09-04 19:00:20,469]::3816500081::INFO::Global best checkpoint updated: seed=42, epoch=1, MAE=0.151227, 1-MAE=0.848773
[2026-09-04 19:00:20,470]::3816500081::DEBUG::Seed: 42 , Epoch: 1 , 1-MAE: 0.8488 , seed best 1-MAE: 0.8488 , patience: 0/20 , 1-Training Loss: 0.8392 , 1-val_avg_o_loss: 0.9018 , 1-val_avg_c_loss: 0.9032 , 1-val_avg_e_loss: 0.8188 , 1-val_avg_a_loss: 0.7837 , 1-val_avg_n_loss: 0.8364
Seed 42 Val Epoch 2: 100%|██████████| 498/498 [00:46<00:00, 10.73it/s]
[2026-09-04 19:

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 123 Val Epoch 1: 100%|██████████| 498/498 [00:47<00:00, 10.51it/s]
[2026-09-05 00:43:10,437]::3816500081::DEBUG::Seed: 123 , Epoch: 1 , 1-MAE: 0.8832 , seed best 1-MAE: 0.8832 , patience: 0/20 , 1-Training Loss: 0.8378 , 1-val_avg_o_loss: 0.8986 , 1-val_avg_c_loss: 0.8909 , 1-val_avg_e_loss: 0.9026 , 1-val_avg_a_loss: 0.8714 , 1-val_avg_n_loss: 0.8528
Seed 123 Val Epoch 2: 100%|██████████| 498/498 [00:47<00:00, 10.43it/s]
[2026-09-05 00:50:40,759]::3816500081::DEBUG::Seed: 123 , Epoch: 2 , 1-MAE: 0.9016 , seed best 1-MAE: 0.9016 , patience: 0/20 , 1-Training Loss: 0.8825 , 1-val_avg_o_loss: 0.9026 , 1-val_avg_c_loss: 0.9051 , 1-val_avg_e_loss: 0.9018 , 1-val_avg_a_loss: 0.8943 , 1-val_avg_n_loss: 0.9041
Seed 123 Val Epoch 3: 100%|██████████| 498/498 [00:47<00:00, 10.46it/s]
[2026-09-05 00:58:10,874]::3816500081::DEBUG::Seed: 123 , Epoch: 3 , 1-MAE: 0.9059 , seed best 1-MAE: 0.9059 , patience: 0/20 , 1-Training Loss: 0.8979 , 1-val_avg_o_loss: 0.8974 , 1-val_avg_c_loss: 0.9080 , 1-

...

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 1004 Val Epoch 1: 100%|██████████| 498/498 [00:47<00:00, 10.58it/s]
[2026-09-05 11:51:57,293]::3816500081::DEBUG::Seed: 1004 , Epoch: 1 , 1-MAE: 0.8509 , seed best 1-MAE: 0.8509 , patience: 0/20 , 1-Training Loss: 0.8388 , 1-val_avg_o_loss: 0.8643 , 1-val_avg_c_loss: 0.8387 , 1-val_avg_e_loss: 0.7645 , 1-val_avg_a_loss: 0.8823 , 1-val_avg_n_loss: 0.9046
Seed 1004 Val Epoch 2: 100%|██████████| 498/498 [00:47<00:00, 10.58it/s]
[2026-09-05 11:59:28,063]::3816500081::DEBUG::Seed: 1004 , Epoch: 2 , 1-MAE: 0.8823 , seed best 1-MAE: 0.8823 , patience: 0/20 , 1-Training Loss: 0.8819 , 1-val_avg_o_loss: 0.8911 , 1-val_avg_c_loss: 0.8498 , 1-val_avg_e_loss: 0.8843 , 1-val_avg_a_loss: 0.8815 , 1-val_avg_n_loss: 0.9047
Seed 1004 Val Epoch 3: 100%|██████████| 498/498 [00:47<00:00, 10.59it/s]
[2026-09-05 12:06:59,089]::3816500081::DEBUG::Seed: 1004 , Epoch: 3 , 1-MAE: 0.8946 , seed best 1-MAE: 0.8946 , patience: 0/20 , 1-Training Loss: 0.8978 , 1-val_avg_o_loss: 0.9082 , 1-val_avg_c_loss: 0.897

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 2024 Val Epoch 1: 100%|██████████| 498/498 [00:47<00:00, 10.56it/s]
[2026-09-05 17:37:26,223]::3816500081::DEBUG::Seed: 2024 , Epoch: 1 , 1-MAE: 0.8087 , seed best 1-MAE: 0.8087 , patience: 0/20 , 1-Training Loss: 0.8398 , 1-val_avg_o_loss: 0.8191 , 1-val_avg_c_loss: 0.8521 , 1-val_avg_e_loss: 0.8470 , 1-val_avg_a_loss: 0.6498 , 1-val_avg_n_loss: 0.8754
Seed 2024 Val Epoch 2: 100%|██████████| 498/498 [00:47<00:00, 10.54it/s]
[2026-09-05 17:44:56,544]::3816500081::DEBUG::Seed: 2024 , Epoch: 2 , 1-MAE: 0.8648 , seed best 1-MAE: 0.8648 , patience: 0/20 , 1-Training Loss: 0.8811 , 1-val_avg_o_loss: 0.9050 , 1-val_avg_c_loss: 0.8629 , 1-val_avg_e_loss: 0.9107 , 1-val_avg_a_loss: 0.8235 , 1-val_avg_n_loss: 0.8221
Seed 2024 Val Epoch 3: 100%|██████████| 498/498 [00:47<00:00, 10.52it/s]
[2026-09-05 17:52:27,159]::3816500081::DEBUG::Seed: 2024 , Epoch: 3 , 1-MAE: 0.9037 , seed best 1-MAE: 0.9037 , patience: 0/20 , 1-Training Loss: 0.8964 , 1-val_avg_o_loss: 0.9033 , 1-val_avg_c_loss: 0.906


MULTI-SEED RESULTS
Seed 42 | Best Epoch  26 | MAE 0.084728 | 1-MAE 0.915272
Seed 123 | Best Epoch  25 | MAE 0.084364 | 1-MAE 0.915636
Seed 777 | Best Epoch  24 | MAE 0.085003 | 1-MAE 0.914997
Seed 1004 | Best Epoch  26 | MAE 0.085403 | 1-MAE 0.914597
Seed 2024 | Best Epoch   9 | MAE 0.085162 | 1-MAE 0.914838
----------------------------------------------------------------------
MAE    : 0.084932 ± 0.000401
1-MAE  : 0.915068 ± 0.000401
Global best checkpoint: seed=123, epoch=25, MAE=0.084364, 1-MAE=0.915636
Saved checkpoint: ./save_swintransformer_folder/model_best_multiseed.pth


In [15]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))